## Example 1: Basic Input and Validation

In [ ]:
import re

from prompt_toolkit.completion import CompleteEvent, WordCompleter
from prompt_toolkit.document import Document
from prompt_toolkit.validation import ValidationError, Validator

# Example 1: Basic Input and Validation
print("Example 1 - Basic Input and Validation")

# Email validator
class EmailValidator(Validator):
    def validate(self, document):
        text = document.text
        if not re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', text):
            raise ValidationError(message='Please enter a valid email address')

# Number validator
class NumberValidator(Validator):
    def __init__(self, min_val=None, max_val=None):
        self.min_val = min_val
        self.max_val = max_val
    
    def validate(self, document):
        text = document.text
        try:
            num = int(text)
            if self.min_val and num < self.min_val:
                raise ValidationError(message=f'Number must be at least {self.min_val}')
            if self.max_val and num > self.max_val:
                raise ValidationError(message=f'Number must be at most {self.max_val}')
        except ValueError:
            raise ValidationError(message='Please enter a valid number')

# A prompt passes its validator a Document; build the same objects directly.
# Each entry: (text, expected to be valid)
test_inputs = {
    "valid_email": ("user@example.com", True),
    "invalid_email": ("invalid-email", False),
    "valid_number": ("25", True),
    "invalid_number": ("abc", False),
    "out_of_range": ("150", False),
}

email_validator = EmailValidator()
number_validator = NumberValidator(min_val=1, max_val=100)

print("\nValidation testing:")
for input_type, (test_input, expected_valid) in test_inputs.items():
    validator = email_validator if "email" in input_type else number_validator
    try:
        validator.validate(Document(test_input))
        valid = True
        print(f"  ✓ {input_type}: '{test_input}' - Valid")
    except ValidationError as e:
        valid = False
        print(f"  ✗ {input_type}: '{test_input}' - {e.message}")
    assert valid == expected_valid, (input_type, test_input)

# Auto-completion
programming_languages = ['Python', 'JavaScript', 'Java', 'C++', 'Go', 'Rust', 'TypeScript']
language_completer = WordCompleter(programming_languages, ignore_case=True)
completions = [c.text for c in language_completer.get_completions(Document("ja"), CompleteEvent())]
print(f"\nCompletions for 'ja': {completions}")
assert completions == ["JavaScript", "Java"], completions

print("Example 1 completed successfully")

## Example 2: Styled Prompts and Formatting

In [ ]:
from prompt_toolkit.formatted_text import ANSI, HTML, fragment_list_to_text, to_formatted_text
from prompt_toolkit.output import DummyOutput
from prompt_toolkit.shortcuts import print_formatted_text
from prompt_toolkit.styles import Style

# Example 2: Styled Prompts and Formatting
print("Example 2 - Styled Prompts and Formatting")

# Define custom styles
custom_style = Style.from_dict({
    'title': '#00aa00 bold',
    'subtitle': '#0077ff italic',
    'error': '#ff0066 bold',
    'success': '#00aa00',
    'warning': '#ff8800',
    'info': '#0077ff',
    'prompt': '#aa00aa bold',
    'user-input': '#ffffff bg:#333333'
})

# A style maps class names to text attributes
print("\n1. Style attributes:")
for name in ('title', 'subtitle', 'error', 'user-input'):
    attrs = custom_style.get_attrs_for_style_str(f'class:{name}')
    print(f"  {name}: color={attrs.color} bg={attrs.bgcolor or '-'} bold={attrs.bold} italic={attrs.italic}")
title_attrs = custom_style.get_attrs_for_style_str('class:title')
assert (title_attrs.color, title_attrs.bold) == ('00aa00', True), title_attrs
assert custom_style.get_attrs_for_style_str('class:subtitle').italic
assert custom_style.get_attrs_for_style_str('class:user-input').bgcolor == '333333'

# HTML-like markup becomes (style, text) fragments
html_examples = [
    '<title>Welcome to the Application</title>',
    '<subtitle>Configuration Setup</subtitle>',
    '<error>✗ Invalid input detected</error>',
    '<success>✓ Operation completed successfully</success>',
    '<warning>⚠ Warning: Check your settings</warning>',
    '<info>ℹ Information: Processing data...</info>'
]

print("\n2. HTML-formatted text:")
for example in html_examples:
    tag = example[1:example.index('>')]
    fragments = to_formatted_text(HTML(example))
    text = fragment_list_to_text(fragments)
    print(f"  {tag:8} -> {text}")
    assert {style for style, _ in fragments} == {f'class:{tag}'}, fragments
    assert text == example[len(tag) + 2:-(len(tag) + 3)], text

# ANSI escape sequences are parsed into styled fragments too
ansi_fragments = to_formatted_text(ANSI('\033[91mRed text\033[0m and plain text'))
print(f"\n3. ANSI text: {fragment_list_to_text(ansi_fragments)!r}")
assert fragment_list_to_text(ansi_fragments) == 'Red text and plain text'
assert ansi_fragments[0][0] == 'ansibrightred', ansi_fragments[0]

# Prompt messages
prompt_templates = {
    'login': HTML('<prompt>Username</prompt> <info>(enter your email)</info>: '),
    'password': HTML('<prompt>Password</prompt> <warning>(hidden)</warning>: '),
    'confirm': HTML('<success>Confirm</success> <subtitle>(y/n)</subtitle>: '),
    'number': HTML('<prompt>Enter number</prompt> <error>(1-100)</error>: ')
}

print("\n4. Styled prompt messages:")
for template_name, template in prompt_templates.items():
    print(f"  {template_name}: {fragment_list_to_text(to_formatted_text(template))!r}")
assert fragment_list_to_text(to_formatted_text(prompt_templates['number'])) == 'Enter number (1-100): '

# HTML.format() escapes its arguments, so values cannot inject markup
toolbar = HTML('<success>Connected</success> | <subtitle>Last saved: {}</subtitle>').format('<12:00>')
print(f"\n5. Toolbar text: {fragment_list_to_text(to_formatted_text(toolbar))!r}")
assert fragment_list_to_text(to_formatted_text(toolbar)) == 'Connected | Last saved: <12:00>'

# Rendering needs no terminal when the output is given explicitly
print_formatted_text(prompt_templates['login'], style=custom_style, output=DummyOutput())

print("Example 2 completed successfully")

## Example 3: Auto-completion and History

In [ ]:
import os
import tempfile

from prompt_toolkit.auto_suggest import AutoSuggestFromHistory
from prompt_toolkit.buffer import Buffer
from prompt_toolkit.completion import CompleteEvent, NestedCompleter, PathCompleter, WordCompleter
from prompt_toolkit.document import Document
from prompt_toolkit.history import FileHistory, InMemoryHistory

# Example 3: Auto-completion and History
print("Example 3 - Auto-completion and History")

def complete(completer, text):
    """Completion texts a completer offers for `text` with the cursor at the end."""
    return [c.text for c in completer.get_completions(Document(text), CompleteEvent())]

languages = ['Python', 'JavaScript', 'TypeScript', 'Java', 'C++', 'Go', 'Rust', 'Swift']
databases = ['PostgreSQL', 'MySQL', 'SQLite', 'MongoDB', 'Redis', 'Elasticsearch']
commands = ['create', 'read', 'update', 'delete', 'list', 'search', 'help', 'exit']
language_completer = WordCompleter(languages, ignore_case=True)
db_completer = WordCompleter(databases, ignore_case=True)
command_completer = WordCompleter(commands, ignore_case=True)

# Nested completion for multi-word commands
nested_commands = {
    'user': {
        'create': WordCompleter(['--name', '--email', '--role']),
        'delete': WordCompleter(['--id', '--username', '--force']),
        'list': WordCompleter(['--filter', '--sort', '--limit']),
        'update': WordCompleter(['--id', '--name', '--email'])
    },
    'database': {
        'connect': WordCompleter(['--host', '--port', '--username', '--password']),
        'backup': WordCompleter(['--output', '--compress', '--exclude']),
        'restore': WordCompleter(['--file', '--overwrite', '--verbose'])
    },
    'system': {
        'status': WordCompleter(['--detailed', '--json', '--watch']),
        'restart': WordCompleter(['--force', '--graceful']),
        'logs': WordCompleter(['--tail', '--follow', '--level'])
    }
}
nested_completer = NestedCompleter.from_nested_dict(nested_commands)

print("\n1. Completion:")
cases = [
    (language_completer, "py", ["Python"]),
    (language_completer, "java", ["JavaScript", "Java"]),
    (db_completer, "postgre", ["PostgreSQL"]),
    (command_completer, "cr", ["create"]),
    (nested_completer, "user c", ["create"]),
    (nested_completer, "database ", ["connect", "backup", "restore"]),
]
for completer, text, expected in cases:
    got = complete(completer, text)
    print(f"  {text!r} -> {got}")
    assert got == expected, (text, got)

sample_commands = [
    "user create --name john --email john@example.com",
    "database connect --host localhost --port 5432",
    "system status --detailed",
    "user list --filter active",
    "database backup --output /tmp/backup.sql"
]

with tempfile.TemporaryDirectory() as temp_dir:
    # Path completion offers the rest of each matching file name
    for name in ("alpha.txt", "alpine.cfg", "beta.txt"):
        open(os.path.join(temp_dir, name), "w").close()
    path_completer = PathCompleter(get_paths=lambda: [temp_dir])
    path_matches = sorted(complete(path_completer, "alp"))
    print(f"  path 'alp' -> {path_matches}")
    assert path_matches == ["ha.txt", "ine.cfg"], path_matches

    print("\n2. History:")
    memory_history = InMemoryHistory()
    for cmd in sample_commands:
        memory_history.append_string(cmd)
    assert list(memory_history.get_strings()) == sample_commands  # oldest first
    print(f"  In-memory history: {len(sample_commands)} commands")

    # FileHistory stores entries in its own format and loads them back, newest first
    history_file = os.path.join(temp_dir, 'command_history.txt')
    file_history = FileHistory(history_file)
    for cmd in sample_commands:
        file_history.append_string(cmd)
    reloaded = list(FileHistory(history_file).load_history_strings())
    assert reloaded == sample_commands[::-1], reloaded
    print(f"  File history reloaded: {len(reloaded)} commands")

# Auto-suggestion completes the current input from the most recent matching history entry
buffer = Buffer(history=memory_history)
suggestion = AutoSuggestFromHistory().get_suggestion(buffer, Document("database b"))
print(f"\n3. Suggestion for 'database b': {suggestion.text!r}")
assert suggestion.text == "ackup --output /tmp/backup.sql", suggestion

# History search
print("\n4. History search:")
for search_term, expected in [("user", 2), ("database", 2), ("--detailed", 1)]:
    matches = [s for s in memory_history.get_strings() if search_term in s]
    print(f"  Search '{search_term}': {len(matches)} matches")
    assert len(matches) == expected, (search_term, matches)

print("Example 3 completed successfully")

## Example 4: Key Bindings and Shortcuts

In [ ]:
from prompt_toolkit.filters import Condition
from prompt_toolkit.key_binding import KeyBindings, merge_key_bindings
from prompt_toolkit.key_binding.defaults import load_key_bindings
from prompt_toolkit.keys import Keys

# Example 4: Key Bindings and Shortcuts
print("Example 4 - Key Bindings and Shortcuts")

kb = KeyBindings()
pressed = []

@kb.add("c-t")
def insert_timestamp(event):
    pressed.append("c-t")

@kb.add("escape", "f")  # Alt-f / Meta-f arrives as Escape followed by f
def forward_word(event):
    pressed.append("escape f")

document_modified = False

@kb.add("c-x", "c-s", filter=Condition(lambda: document_modified))
def save_document(event):
    pressed.append("c-x c-s")

# Look bindings up by key sequence, as the key processor does
print("\n1. Registered bindings:")
for keys in [("c-t",), (Keys.ControlT,), (Keys.Escape, "f"), (Keys.ControlX, Keys.ControlS)]:
    bindings = kb.get_bindings_for_keys(keys)
    print(f"  {keys} -> {[b.handler.__name__ for b in bindings]}")
    assert bindings, keys
assert kb.get_bindings_for_keys(("c-t",))[0].handler is insert_timestamp

# After Ctrl-X, the registry knows a longer sequence can follow
prefixed = kb.get_bindings_starting_with_keys((Keys.ControlX,))
assert [b.handler for b in prefixed] == [save_document], prefixed

# A filter is evaluated when the key is pressed
save_binding = kb.get_bindings_for_keys((Keys.ControlX, Keys.ControlS))[0]
assert not save_binding.filter()
document_modified = True
assert save_binding.filter()
print("\n2. Ctrl-X Ctrl-S is active only while the document is modified")

# Handlers are plain callables that receive the key-press event
kb.get_bindings_for_keys(("c-t",))[0].handler(None)
assert pressed == ["c-t"], pressed

# The default Emacs-style editing bindings merge with custom ones
merged = merge_key_bindings([load_key_bindings(), kb])
assert merged.get_bindings_for_keys((Keys.ControlE,)), "default Ctrl-E (end of line) binding missing"
assert insert_timestamp in [b.handler for b in merged.get_bindings_for_keys((Keys.ControlT,))]
print(f"\n3. Default plus custom bindings: {len(merged.bindings)}")

print("Example 4 completed successfully")

## Example 5: An Application Driven by Pipe Input

In [ ]:
from prompt_toolkit.application import Application
from prompt_toolkit.input import create_pipe_input
from prompt_toolkit.key_binding import KeyBindings
from prompt_toolkit.layout import HSplit, Layout, VSplit
from prompt_toolkit.output import DummyOutput
from prompt_toolkit.widgets import Box, Button, Frame, Label, TextArea

# Example 5: a real Application, driven by pipe input instead of a terminal
print("Example 5 - Application on pipe input")

text_area = TextArea(text="hi", multiline=False)
ok_button = Button("OK")
status = Label("status: ready")
root = Box(Frame(HSplit([text_area, VSplit([ok_button, status])]), title="Demo"))
layout = Layout(root, focused_element=text_area)

kb = KeyBindings()

@kb.add("c-q")
def quit_app(event):
    event.app.exit(result=text_area.text)

with create_pipe_input() as pipe_input:
    app = Application(layout=layout, key_bindings=kb, input=pipe_input, output=DummyOutput())

    windows = list(app.layout.find_all_windows())
    print(f"Layout windows: {len(windows)}")
    assert text_area.window in windows and ok_button.window in windows
    assert app.layout.has_focus(text_area)

    # Ctrl-E (end of line), type " there", then Ctrl-Q, which exits with the text area's content.
    # in_thread=True gives the application its own event loop, so this also works in a notebook.
    pipe_input.send_text("\x05 there\x11")
    result = app.run(in_thread=True)

print(f"Application result: {result!r}")
assert result == "hi there", result

print("Example 5 completed successfully")